# 05. Dates and times

**Tier:** Intermediate. **Why you'd use this:** A meeting at 09:00 in New York is not the same instant as 09:00 with no zone. Naive datetimes hide that bug until someone travels.

**After this lesson you can:** build `datetime` values; attach a zone with `zoneinfo`; avoid comparing naive and aware times.
About 25 minutes. **Uses:** [Classes](../11-classes/01-classes-lesson.ipynb)


## Quick review

1. What is the difference between `str` and `bytes`?
2. Where should encoding happen?

<details><summary>Answers</summary>

Text versus raw octets. At the program boundary.
</details>


## The idea

`datetime` stores a calendar date and clock time. Without a time zone it is naive: numbers on a wall clock with no location. `zoneinfo.ZoneInfo` makes it aware. Compare aware to aware. Equality with a naive value is silently `False`; ordering raises.

<details><summary>Explain more</summary>

`datetime(2024, 1, 1, 9, 0, tzinfo=ZoneInfo("America/New_York"))` is an instant you can convert to UTC.
</details>



## See it work

Stamp a time in a named zone.


In [1]:
from datetime import datetime
from zoneinfo import ZoneInfo

# 1. Aware datetime: clock plus zone.
meeting = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("America/New_York"))
print(meeting)
print(meeting.tzinfo)
print(meeting.astimezone(ZoneInfo("UTC")))

2024-06-01 09:00:00-04:00
America/New_York
2024-06-01 13:00:00+00:00


## Check yourself

Answer these before you run the cell.

1. What is missing from a naive datetime?
2. What does `naive == aware` return? What about `naive < aware`?


In [2]:
from datetime import datetime
from zoneinfo import ZoneInfo

naive = datetime(2024, 6, 1, 9, 0)
aware = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("UTC"))
print(naive.tzinfo)
print(aware.tzinfo)

None
UTC


<details><summary>Answers</summary>

A time zone. Equality returns False with no error. Ordering (<, >) raises TypeError. The silent False is the dangerous case.
</details>


## Why it works this way

Wall clocks need a time zone to become instants. A naive datetime is only a civil date and time. An aware datetime carries `tzinfo` so conversions and comparisons have meaning across regions. `zoneinfo` ships the IANA database names like `America/New_York`.

Equality between naive and aware is silently `False`. Ordering (`<`, `>`) raises `TypeError`. The silent `False` is worse than a crash: code can treat two unrelated clocks as "not equal" and move on. Storing UTC internally and converting for display is a common pattern. DST transitions mean "add one day" is not always twenty-four hours; calendar methods exist for civil arithmetic.

Cost of zone lookups is small after the database loads. The design reason for awareness is correctness across regions. Evidence: printing `tzinfo`, converting with `astimezone`, equality printing `False`, and `TypeError` on ordering.

Never assume local time equals UTC. Never serialize timestamps without a zone or offset if they represent instants. APIs that return naive datetimes force you to ask which zone they meant. Prefer APIs that include offsets. In logs, ISO strings with offsets beat ambiguous local stamps.

Unix timestamps are instants. Civil dates like birthdays are not. Choose representations that match the meaning. Libraries that ignore zones push bugs into your business logic. When testing, freeze time with known aware values instead of calling `now()` and hoping. Serialize with offsets or `Z` for UTC so readers do not guess. Database columns often store UTC. UI layers convert to the viewer zone. Keep that conversion out of your core math when you can.


In [3]:
from datetime import datetime
from zoneinfo import ZoneInfo

meeting = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("America/New_York"))
print(meeting)
print(meeting.astimezone(ZoneInfo("UTC")))
naive = datetime(2024, 6, 1, 9, 0)
print(naive.tzinfo)

2024-06-01 09:00:00-04:00
2024-06-01 13:00:00+00:00
None


In [4]:
from datetime import datetime
from zoneinfo import ZoneInfo

naive = datetime(2024, 6, 1, 9, 0)
aware = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("UTC"))
print(naive == aware)

False


In [5]:
from datetime import datetime
from zoneinfo import ZoneInfo

naive = datetime(2024, 6, 1, 9, 0)
aware = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("UTC"))
print(naive < aware)

TypeError: can't compare offset-naive and offset-aware datetimes

## When to use it

Use aware datetimes for instants. Use naive only for pure civil dates without instants, and say so. Store UTC or offset-aware values at boundaries. Don't compare naive and aware. Don't hard-code fixed offsets when a named zone is available. Store instants with zones; convert for display at the UI edge.


## Common mistake

Treating `naive == aware` as a safe check. It returns `False` without raising, so a branch that meant "same instant" can silently take the wrong path.


In [6]:
from datetime import datetime
from zoneinfo import ZoneInfo

naive = datetime(2024, 6, 1, 9, 0)
aware = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("UTC"))
same_clock = naive == aware  # looks like a check; returns False
print(same_clock)

False


Both say 09:00 on the same calendar day, but one has no zone. Equality does not raise. Attach a zone before you compare, or compare UTC instants.


In [7]:
from datetime import datetime
from zoneinfo import ZoneInfo

naive = datetime(2024, 6, 1, 9, 0)
aware = datetime(2024, 6, 1, 9, 0, tzinfo=ZoneInfo("UTC"))
naive_as_utc = naive.replace(tzinfo=ZoneInfo("UTC"))
print(naive_as_utc == aware)

True


## Try it

Convert a New York meeting to `Europe/Berlin`. Does the calendar hour change when zones differ?


## Recap

- Naive datetimes have no zone. Aware ones do.
- Use `zoneinfo.ZoneInfo` for named zones.
- `naive == aware` is silently `False`; ordering raises `TypeError`.

**Next:** [practice](05-dates-and-times-practice.ipynb)
